<a href="https://colab.research.google.com/github/Dewatcher/cifar10-animal-object-recognizer/blob/dev/notebooks/01_baseline_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image
import numpy as np

# 1. Define standard CIFAR-10 classes and ADAS Hazard Mapping
CIFAR10_CLASSES = (
    'plane', 'car', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
)

ADAS_HAZARD_MAP = {
    'plane': 'Low-Risk / Contextual Asset',
    'car': 'Vehicle Hazard (Proximity Tracking)',
    'bird': 'Living Threat (Emergency Warning)',
    'cat': 'Living Threat (Emergency Warning)',
    'deer': 'Living Threat (Emergency Warning)',
    'dog': 'Living Threat (Emergency Warning)',
    'frog': 'Living Threat (Emergency Warning)',
    'horse': 'Living Threat (Emergency Warning)',
    'ship': 'Low-Risk / Contextual Asset',
    'truck': 'Vehicle Hazard (Proximity Tracking)'
}

# 2. Set Device (GPU if available)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Load CIFAR-10 Test Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=True)

# 4. Load Pretrained ResNet-18 Model & Modify Final Layer for 10 Classes
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Replace ResNet's final FC layer (1000 ImageNet outputs) with 10 CIFAR outputs
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 10)

model = model.to(device)
model.eval()

# 5. Grab ONE Sample Image & Run Prediction
dataiter = iter(testloader)
images, labels = next(dataiter)
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    probabilities = torch.nn.functional.softmax(outputs, dim=1)
    confidence, predicted = torch.max(probabilities, 1)

actual_class = CIFAR10_CLASSES[labels[0].item()]
predicted_class = CIFAR10_CLASSES[predicted[0].item()]
conf_score = confidence[0].item() * 100
hazard_type = ADAS_HAZARD_MAP[predicted_class]

# 6. Display Baseline Result
print("\n" + "="*45)
print(" BASELINE PREDICTION RESULT")
print("="*45)
print(f"Ground Truth Class : {actual_class.upper()}")
print(f"Predicted Class    : {predicted_class.upper()}")
print(f"Confidence Score   : {conf_score:.2f}%")
print(f"ADAS Hazard Status : {hazard_type}")
print("="*45)
print(" Foundation verified successfully!")

Using device: cuda:0


100%|██████████| 170M/170M [22:12<00:00, 128kB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 145MB/s]



 BASELINE PREDICTION RESULT
Ground Truth Class : BIRD
Predicted Class    : CAR
Confidence Score   : 17.57%
ADAS Hazard Status : Vehicle Hazard (Proximity Tracking)
 Foundation verified successfully!


In [2]:
import torch.nn as nn
import torch.optim as optim

# 1. Load CIFAR-10 Training Set
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# 2. Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)  # Train only the new final layer

# 3. Fine-Tune for 3 Epochs
epochs = 3
print("Starting Fine-Tuning...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch + 1}/{epochs}] - Loss: {running_loss / len(trainloader):.4f}")

print("Fine-tuning complete!")

# 4. Save Trained Weights for Streamlit / Azure Deployment
torch.save(model.state_dict(), 'cifar10_resnet18_adas.pth')
print("Model weights saved to 'cifar10_resnet18_adas.pth'")

Starting Fine-Tuning...
Epoch [1/3] - Loss: 1.7257
Epoch [2/3] - Loss: 1.5955
Epoch [3/3] - Loss: 1.5779
Fine-tuning complete!
Model weights saved to 'cifar10_resnet18_adas.pth'


In [3]:
model.eval()

# Grab a new test sample
dataiter = iter(testloader)
images, labels = next(dataiter)
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    probabilities = torch.nn.functional.softmax(outputs, dim=1)
    confidence, predicted = torch.max(probabilities, 1)

actual_class = CIFAR10_CLASSES[labels[0].item()]
predicted_class = CIFAR10_CLASSES[predicted[0].item()]
conf_score = confidence[0].item() * 100
hazard_type = ADAS_HAZARD_MAP[predicted_class]

print("\n" + "="*45)
print(" POST-FINE-TUNING PREDICTION RESULT")
print("="*45)
print(f"Ground Truth Class : {actual_class.upper()}")
print(f"Predicted Class    : {predicted_class.upper()}")
print(f"Confidence Score   : {conf_score:.2f}%")
print(f"ADAS Hazard Status : {hazard_type}")
print("="*45)


 POST-FINE-TUNING PREDICTION RESULT
Ground Truth Class : BIRD
Predicted Class    : BIRD
Confidence Score   : 28.79%
ADAS Hazard Status : Living Threat (Emergency Warning)
